# Embeddings

Um *embedding* é uma função que leva objetos discretos e de estrutura complicada, como palavras, frases, imagens ou nós de um grafo, para vetores densos de um espaço $\mathbb{R}^d$, de modo que a proximidade geométrica no destino corresponda a alguma noção de semelhança na origem. Uma vez que os dados estão nessa forma, todo o ferramental de aprendizado não supervisionado passa a se aplicar a eles: é possível medir distâncias, agrupar, projetar e buscar. Este notebook constrói embeddings de texto a partir de um modelo pré-treinado, verifica passo a passo como eles são produzidos e examina o que a geometria resultante de fato garante.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
import torch
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModel, AutoTokenizer

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

As bibliotecas `transformers` e `sentence-transformers` não fazem parte do scikit-learn e precisam ser instaladas à parte com `pip install transformers sentence-transformers`. Na primeira execução, os pesos dos modelos são baixados e guardados em cache local.

## Fundamentação

Formalmente, um embedding é um mapa $f: \mathcal{X} \to \mathbb{R}^d$ tal que, para uma relação de similaridade $\text{sim}$ definida sobre $\mathcal{X}$,

$$
\text{sim}(x_1, x_2) \text{ alta} \iff f(x_1) \text{ e } f(x_2) \text{ próximos em } \mathbb{R}^d
$$

O que muda entre aplicações é como esse mapa é obtido e o que exatamente "próximo" significa.

### Por Que Não One-Hot

A representação ingênua de um vocabulário de $V$ palavras é o vetor indicador: a palavra $i$ vira o vetor com 1 na posição $i$ e 0 nas demais. Ela tem duas propriedades ruins. A dimensão é $V$, na casa das centenas de milhares. E, pior, **quaisquer duas palavras distintas são equidistantes**: o produto interno entre dois vetores indicadores diferentes é sempre 0, então "gato" está tão longe de "cachorro" quanto de "hipoteca". Toda a informação semântica é descartada por construção.

Um embedding troca essa representação esparsa de dimensão $V$ por uma densa de dimensão $d \ll V$, na qual as coordenadas não têm significado individual mas as relações entre vetores carregam informação.

### Similaridade de Cosseno

A medida usual de proximidade entre embeddings é o cosseno do ângulo entre os vetores:

$$
\cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|\,\|\mathbf{v}\|}
$$

O valor varia entre $-1$ e $1$. A escolha do cosseno em vez da distância euclidiana não é arbitrária: a norma de um embedding de texto costuma refletir características como o comprimento da frase ou a frequência dos seus termos, e não o seu conteúdo. Normalizar remove esse fator. Note que, para vetores de norma 1, as duas medidas são equivalentes, já que $\|\mathbf{u} - \mathbf{v}\|^2 = 2 - 2\,\mathbf{u} \cdot \mathbf{v}$: minimizar a distância euclidiana e maximizar o cosseno passam a ser o mesmo problema.

## Do Texto ao Vetor

Um modelo Transformer não produz um vetor por frase, e sim um vetor por *token*. Para chegar a uma representação única da frase é preciso agregar esses vetores. Vamos percorrer o processo explicitamente antes de usar a biblioteca que o encapsula.

In [ ]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
encoder = AutoModel.from_pretrained(model_name).eval()

sentences = [
    "Natural language processing has advanced greatly.",
    "Artificial intelligence is transforming the world.",
    "What is the capital of France?",
    "Paris is the most populated city in France.",
]

encoded = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')
print("input_ids:", tuple(encoded['input_ids'].shape))
print("tokens da primeira frase:", tokenizer.convert_ids_to_tokens(encoded['input_ids'][0]))

O tokenizador quebra o texto em subpalavras, acrescenta os marcadores `[CLS]` no início e `[SEP]` no fim, e completa as frases mais curtas com `[PAD]` até que todas tenham o mesmo comprimento. A matriz `attention_mask` registra quais posições são conteúdo real e quais são preenchimento.

In [ ]:
with torch.no_grad():
    token_embeddings = encoder(**encoded).last_hidden_state

print("saída do Transformer:", tuple(token_embeddings.shape), "(frases, tokens, dimensão)")

### Mean Pooling

A agregação mais usada é a média dos vetores dos tokens, **ponderada pela máscara** para que as posições de preenchimento não entrem na conta. O resultado é normalizado para norma 1, o que faz do produto interno a própria similaridade de cosseno.

In [ ]:
def mean_pooling(token_embeddings, attention_mask):
    """Média dos vetores de token, ignorando as posições de padding."""
    mask = attention_mask.unsqueeze(-1).float()
    return (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)


pooled = mean_pooling(token_embeddings, encoded['attention_mask'])
manual = torch.nn.functional.normalize(pooled, p=2, dim=1).numpy()

print("embedding da frase:", manual.shape)
print("normas:", np.round(np.linalg.norm(manual, axis=1), 6))

### A Biblioteca sentence-transformers

O pacote `sentence-transformers` empacota tokenização, passagem pelo Transformer, pooling e normalização em uma única chamada. Podemos confirmar que ele faz exatamente o que acabamos de fazer à mão.

In [ ]:
model = SentenceTransformer(model_name)
library = model.encode(sentences)

print(f"dimensão do embedding: {library.shape[1]}")
print(f"maior diferença absoluta em relação ao cálculo manual: {np.abs(manual - library).max():.2e}")

A diferença máxima é da ordem de $10^{-7}$, ou seja, apenas erro de arredondamento em ponto flutuante de 32 bits. Os dois caminhos produzem o mesmo vetor.

Vale ver que a escolha do pooling não é um detalhe. Uma alternativa comum é usar apenas o vetor do token `[CLS]`, que em modelos de classificação recebe a representação agregada da sequência.

In [ ]:
cls = torch.nn.functional.normalize(token_embeddings[:, 0], p=2, dim=1).numpy()

for sentence, similarity in zip(sentences, (cls * manual).sum(axis=1)):
    print(f"cos(CLS, mean) = {similarity:.4f}   {sentence}")

Os dois pooling produzem vetores bem diferentes, com cossenos entre 0,30 e 0,58. O `all-MiniLM-L6-v2` foi treinado com um objetivo contrastivo que otimiza justamente a saída do *mean pooling*; usar o `[CLS]` neste modelo devolve um vetor que não foi treinado para essa finalidade. O pooling faz parte da definição do embedding, não é uma escolha posterior e livre.

## Similaridade Semântica

Com os textos representados como vetores, comparações semânticas viram operações de álgebra linear.

In [ ]:
def cosine(u, v):
    """Similaridade de cosseno entre dois vetores."""
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))


pairs = [
    ("What is the capital of the United Kingdom?", "London is the main city in the UK."),
    ("The stock market experienced a drop today.", "Shares fell sharply during the session."),
    ("I like to cook Italian dishes on Sundays.", "Roasting vegetables caramelizes natural sugars."),
    ("Cats are popular pets.", "Quantum physics is a complex field."),
]

for first, second in pairs:
    a, b = model.encode([first, second])
    print(f"{cosine(a, b):.4f}   {first[:44]:<46} | {second}")

Os quatro pares cobrem uma faixa larga e ordenada. Os dois primeiros, 0,605 e 0,549, são reformulações do mesmo conteúdo com vocabulário quase inteiramente distinto, trocando "capital" por "main city" e "market drop" por "shares fell". O terceiro, 0,166, une dois textos do mesmo domínio (culinária) mas sem relação direta. O último, 0,044, junta dois assuntos sem nada em comum.

Como os embeddings já saem normalizados, o produto interno basta e a função `cosine_similarity` do scikit-learn devolve o mesmo resultado.

In [ ]:
a, b = model.encode(["What is the capital of the United Kingdom?", "London is the main city in the UK."])
print(f"produto interno:      {float(a @ b):.6f}")
print(f"cosine_similarity:    {cosine_similarity([a], [b])[0][0]:.6f}")

## Busca Semântica

A busca por palavras-chave devolve documentos que contêm os termos da consulta. A busca semântica devolve documentos cujo *significado* se aproxima do da consulta, ainda que não compartilhem nenhuma palavra. O procedimento tem três passos: transformar o corpus em vetores uma única vez, transformar a consulta em vetor, e ordenar o corpus pela similaridade com ela.

In [ ]:
corpus = [
    "Government unveils new education reform plan.",
    "Senators clash over digital surveillance legislation.",
    "Elections in Europe mark historic voter turnout.",
    "AI startup releases open-source multimodal model.",
    "Quantum processor achieves record computational speed.",
    "Hackers target major bank in ransomware attack.",
    "Global markets rebound after week of volatility.",
    "Central bank raises rates to combat inflation.",
    "Tech stocks lead gains in morning trading session.",
    "Scientists confirm discovery of Earth-like exoplanet.",
    "New vaccine shows promise against emerging virus.",
    "Researchers achieve milestone in clean energy fusion.",
    "International film festival celebrates women directors.",
    "Museum opens interactive exhibit on digital art.",
    "City launches public program to promote urban cycling.",
]

corpus_embeddings = model.encode(corpus)
print("índice:", corpus_embeddings.shape)

In [ ]:
def search(query, k=3):
    """Retorna os k textos do corpus mais próximos da consulta."""
    scores = corpus_embeddings @ model.encode([query])[0]
    ranking = np.argsort(scores)[::-1][:k]
    print(f"consulta: {query!r}")
    for i in ranking:
        print(f"  {scores[i]:.4f}  {corpus[i]}")


search("Planets similar to Earth")

A resposta correta aparece em primeiro lugar com 0,554, e a segunda colocada fica em 0,072, uma distância enorme. Quando a consulta tem uma resposta única e clara no corpus, a diferença entre o primeiro e o segundo lugar é grande, o que dá uma forma natural de decidir se vale a pena responder.

O caso mais interessante é aquele em que a consulta e a resposta não compartilham nenhuma palavra.

In [ ]:
query = "cyberattack against a financial institution"
search(query)


def word_overlap(a, b):
    """Número de palavras em comum entre dois textos."""
    return len(set(re.findall(r'\w+', a.lower())) & set(re.findall(r'\w+', b.lower())))


best_keyword = max(range(len(corpus)), key=lambda i: word_overlap(query, corpus[i]))
print(f"\npalavras em comum entre a consulta e a resposta correta: "
      f"{word_overlap(query, 'Hackers target major bank in ransomware attack.')}")
print(f"melhor resultado por palavras-chave: {corpus[best_keyword]!r} "
      f"({word_overlap(query, corpus[best_keyword])} palavra em comum)")

A busca semântica acerta com 0,445, apesar de a consulta e a resposta não terem **nenhuma** palavra em comum: "cyberattack" contra "hackers"/"ransomware", "financial institution" contra "bank". Já a busca por sobreposição de palavras retorna uma notícia sobre vacinas, porque foi a única a compartilhar um termo com a consulta, e esse termo é a preposição "against".

## A Geometria do Espaço

Uma pergunta que precede qualquer uso desses vetores: o que a similaridade de cosseno mede, na escala em que ela aparece? Para responder, é preciso saber qual é a similaridade **típica** entre dois textos sem relação nenhuma.

In [ ]:
def pairwise_cosines(embeddings):
    """Cossenos de todos os pares distintos de um conjunto de embeddings."""
    normalized = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    matrix = normalized @ normalized.T
    return matrix[np.triu_indices(len(embeddings), k=1)]


def raw_bert_embeddings(texts, name='bert-base-uncased'):
    """Mean pooling sobre um BERT sem ajuste contrastivo."""
    tok = AutoTokenizer.from_pretrained(name)
    base = AutoModel.from_pretrained(name).eval()
    enc = tok(texts, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        hidden = base(**enc).last_hidden_state
    return mean_pooling(hidden, enc['attention_mask']).numpy()


rng = np.random.default_rng(42)
random_vectors = rng.normal(size=(len(corpus), 384))

reference = {
    'all-MiniLM-L6-v2': pairwise_cosines(corpus_embeddings),
    'BERT sem ajuste': pairwise_cosines(raw_bert_embeddings(corpus)),
    'vetores aleatórios': pairwise_cosines(random_vectors),
}

for name, values in reference.items():
    print(f"{name:>20}: média={values.mean():+.4f}  desvio={values.std():.4f}  "
          f"faixa=[{values.min():+.4f}, {values.max():+.4f}]")

In [ ]:
plt.figure(figsize=(9, 5))
for name, values in reference.items():
    plt.hist(values, bins=20, histtype='step', linewidth=2, label=name)
plt.axvline(0, color='gray', ls='--', lw=1)
plt.xlabel('similaridade de cosseno entre pares quaisquer')
plt.ylabel('frequência')
plt.legend()
plt.title('Distribuição de similaridade entre textos sem relação')
plt.show()

Os três histogramas contam a história completa. Vetores gaussianos aleatórios em 384 dimensões têm cosseno médio $+0{,}003$: em dimensão alta, direções sorteadas ao acaso são praticamente ortogonais. O `all-MiniLM-L6-v2` fica em 0,069, muito perto disso, o que significa que os embeddings ocupam o espaço de forma quase isotrópica e um cosseno de 0,5 realmente indica "bem mais parecido que o normal".

O BERT sem ajuste contrastivo tem média 0,664 e mínimo 0,524. Todos os textos, relacionados ou não, ficam concentrados em um cone estreito do espaço. Esse fenômeno se chama **anisotropia**, e é a razão de o mean pooling sobre um BERT genérico funcionar mal como embedding de sentença: um cosseno de 0,7 não distingue nada, porque 0,66 é o valor de referência.

A consequência prática é que **valores de cosseno só têm significado relativo ao modelo que os produziu**. Um limiar de 0,5 é restritivo em um modelo e permissivo em outro. Antes de escolher qualquer limiar, meça a distribuição de referência para textos que você sabe que não têm relação.

Curiosamente, isso não impede o BERT bruto de ordenar razoavelmente: para a consulta anterior ele também coloca a notícia de ransomware em primeiro lugar. O que ele perde é a *escala*, porque a separação entre o primeiro e os demais colocados fica pequena demais para ser útil como critério de decisão.

## Visualizando o Espaço

Com 384 dimensões, a única forma de olhar para a estrutura do corpus é projetar. Como o corpus é pequeno, usamos uma perplexidade baixa no t-SNE.

In [ ]:
pca = PCA(n_components=2)
projections = {
    f'PCA ({pca.fit(corpus_embeddings).explained_variance_ratio_.sum():.1%} da variância)':
        pca.transform(corpus_embeddings),
    't-SNE (perplexity=5)':
        TSNE(n_components=2, perplexity=5, init='pca', random_state=42).fit_transform(corpus_embeddings),
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (title, Z) in zip(axes, projections.items()):
    ax.scatter(Z[:, 0], Z[:, 1], s=90, color='steelblue')
    for (x, y), text in zip(Z, corpus):
        ax.annotate(text, (x, y), fontsize=8, xytext=(4, 4), textcoords='offset points')
    ax.set(title=title, xticks=[], yticks=[])
plt.tight_layout()
plt.show()

Ambas as projeções aproximam as três notícias de mercado financeiro, e separam a de exoplanetas do resto. Com apenas 15 pontos e 21,2% da variância retida em duas componentes, porém, qualquer leitura fina do gráfico é frágil: a projeção é um ponto de partida para gerar hipóteses, não uma medida.

## Custo e Limitações

Gerar embeddings custa uma passagem pelo Transformer por texto, com custo quadrático no número de tokens por causa da atenção. Na prática, o corpus é vetorizado uma vez e só a consulta é processada a cada busca. A busca em si é uma multiplicação matriz-vetor de custo $O(N d)$, viável até algumas centenas de milhares de documentos; acima disso usam-se índices aproximados de vizinhos, que trocam exatidão por tempo sublinear.

A primeira limitação a considerar é que a escala do cosseno depende do modelo. Como visto, 0,66 é o valor de referência entre textos sem relação em um modelo e 0,00 em outro, de forma que limiares não são transferíveis. Junto com isso, é o modelo que define o que conta como "similar": o `all-MiniLM-L6-v2` foi treinado com pares de textos em inglês, e a noção de similaridade que ele codifica é a daqueles dados, podendo não representar domínios técnicos, jargão específico ou outros idiomas.

Não há garantia de composicionalidade. "O gato perseguiu o cachorro" e "O cachorro perseguiu o gato" produzem vetores quase idênticos, embora afirmem coisas opostas. Embeddings capturam tópico e assunto muito melhor do que estrutura lógica, negação ou papéis sintáticos.

Restam três restrições práticas. O comprimento máximo é limitado, e o `all-MiniLM-L6-v2` trunca em 256 tokens, então textos longos precisam ser divididos em trechos, sendo que a escolha de como dividir afeta o resultado tanto quanto o modelo. Vieses do treino são herdados, já que as relações geométricas refletem as regularidades do corpus de origem, incluindo associações indesejadas. E as dimensões não têm significado individual: não há como inspecionar a coordenada 137 e dizer o que ela representa, porque a informação está nas relações entre vetores e não nas coordenadas.

## Exercícios

### Exercício 1: Pooling

Compare três formas de agregar os vetores de token em um vetor de frase: a média com máscara implementada acima, o vetor do token `[CLS]` e o máximo elemento a elemento (*max pooling*). Para cada uma, calcule a similaridade nos quatro pares da seção de similaridade semântica e a distribuição de referência sobre o corpus. Qual delas separa melhor os pares relacionados dos não relacionados?

In [ ]:
# Seu código aqui

### Exercício 2: Negação e Ordem

Monte pelo menos cinco pares de frases que diferem apenas por negação ("The treatment is effective." / "The treatment is not effective.") ou por inversão de papéis ("The dog chased the cat." / "The cat chased the dog."). Meça a similaridade de cosseno de cada par e compare com a distribuição de referência do corpus. O modelo distingue esses pares? Discuta o que isso implica para usar busca semântica em um domínio onde a negação importa, como bulas de medicamentos ou cláusulas contratuais.

In [ ]:
# Seu código aqui

### Exercício 3: Um Índice de Busca Maior

Carregue um corpus com alguns milhares de textos, por exemplo com `sklearn.datasets.fetch_20newsgroups`, e construa um índice de busca semântica sobre ele. Meça o tempo de vetorização do corpus e o tempo médio por consulta. Em seguida, compare os resultados da busca semântica com os de uma busca por TF-IDF (`sklearn.feature_extraction.text.TfidfVectorizer` seguido de `cosine_similarity`) para consultas que usam sinônimos dos termos dos documentos. Onde cada abordagem ganha?

In [ ]:
# Seu código aqui